# Drug Co-occurrence Pattern Mining

Explore frequent itemsets and association rules in recorded overdose data.


In [ ]:
import pandas as pd
import numpy as np
import time

In [ ]:
from pathlib import Path

DATA_PATH = Path("data/accidental_drug_related_deaths.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download the CSV linked in the README."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} records and {len(df.columns)} columns.")


# 2. Understanding The Dataset

In [ ]:
#Configuring Display (for pycharm)
desired_width=5000
pd.set_option('display.width', desired_width)
np.set_printoptions(linewidth=desired_width)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

#Step 1
#inferring data from our data set
print(df.head(5))
print(df.info())
print(df.shape)  #so we know th number of columns

#Step 2
#replace all the NaN (null vals)  with empty
df.replace(np.nan, "", inplace=True)
print(df.head(5))


# 3. Objective Oriented Preprocessing

# 3.1 Dropping Irrelevant Columns and Rows

Run the remaining cells in order so preprocessing and pattern-mining inputs stay consistent.

In [ ]:
#Removing rows and columns unrelated to research objectives
# Define the allowed values for each column
allowed_date_types = ['Date of death']  # Allow only DofDeath for 'Date Type' (not injury or reported)
allowed_sexes = ['Male', 'Female']
allowed_races = ['White', 'Black', 'Black or African American']
allowed_residence_cities = ['HARTFORD', 'NEW HAVEN', 'WATERBURY', 'BRIDGEPORT', 'NEW BRITAIN']
allowed_death_cities = ['HARTFORD', 'NEW HAVEN', 'WATERBURY', 'BRIDGEPORT', 'NEW BRITAIN']
allowed_fentanyl_values = ['Y', '']  # Keep 'Y' and empty values for 'Fentanyl'

# Columns to drop
columns_to_drop = ['DeathCityGeo', 'InjuryCityGeo', 'ResidenceCityGeo', 'Other', 'Any Opioid',
                   'Heroin/Morph/Codeine', 'Opiate NOS', 'Gabapentin', 'Xylazine',
                   'Morphine (Not Heroin)', 'Hydromorphone', 'Tramad', 'Amphet', 'Meth/Amphetamine',
                   'Methadone', 'Benzodiazepine', 'Hydrocodone', 'Ethanol', 'Oxymorphone',
                   'Oxycodone', 'Fentanyl Analogue', 'Heroin death certificate (DC)',
                   'Other Significant Conditions ', 'Cause of Death', 'Manner of Death', 'Location if Other',
                   'Location', 'Death State', 'Death County', 'Description of Injury',
                   'Injury Place', 'Injury State', 'Injury County', 'Injury City',
                   'Residence State', 'Residence County', 'Ethnicity']

# Drop the columns
refined_df = df.drop(columns=columns_to_drop)

# Filter the DataFrame based on the allowed values
death_df = refined_df[
    refined_df['Date Type'].isin(allowed_date_types) &
    refined_df['Sex'].isin(allowed_sexes) &
    refined_df['Residence City'].isin(allowed_residence_cities) &
    refined_df['Death City'].isin(allowed_death_cities) &
    refined_df['Fentanyl'].isin(allowed_fentanyl_values) &
    refined_df['Race'].isin(allowed_races)
]

# Reset the index of the new DataFrame so as to avoid index errors
death_df.reset_index(drop=True, inplace=True)

#Gettting an idea for the unique and missing values in each feature
for column in refined_df.columns:
    print(f"**Column:** `{column}`")
    value_counts = death_df[column].value_counts()
    formatted_values = [f"{value} ({count})" for value, count in value_counts.items()]
    print("  Unique Values:", ", ".join(formatted_values))

    print("-" * 40)
print("Missing Values: \n", death_df.isna().sum())


# 3.2 Dealing with Date

In [ ]:
# Extracting the Year from date for better general anlaysis of broad trends

if 'Date' in death_df.columns:
    # Convert 'Date' column to datetime, handling errors
    death_df['Date'] = pd.to_datetime(death_df['Date'], errors='coerce')

    # Extract year and create a new 'Year' column
    death_df['Year'] = death_df['Date'].dt.year

    # Drop the original 'Date' column (optional, if you don't need it anymore)
    death_df = death_df.drop(columns=['Date'])

# 3.3 Encoding 'Sex' column and Discretizing 'Age' column

In [ ]:
import matplotlib.pyplot as plt

#Merging similar categories and Discretization
#Age and Sex

# Encode 'Sex'
death_df['Sex'] = death_df['Sex'].replace({'Male': 1, 'Female': 0}, inplace = False)

# Ensure 'Age' column is numeric and coerce errors
death_df['Age'] = pd.to_numeric(death_df['Age'], errors='coerce')

# Dropping any rows with 'nan' in 'Age' column before binning
#this way nan values will not be created in Age_Bin column
death_df = death_df[~np.isnan(death_df['Age'])]
death_df.reset_index(drop=True, inplace=True)

# Discretize Age into 10 year bins
min_age = death_df['Age'].min()
max_age = death_df['Age'].max()
bins = np.arange(min_age, max_age + 10, 10)
death_df['Age_Bin'] = pd.cut(death_df['Age'], bins=bins, labels=False, right = False)
print(f"  Unique values in Age_Bin after binning: {death_df['Age_Bin'].unique()}")


# Reset index after dropping rows
death_df.reset_index(drop=True, inplace=True)


# This will print an age histogram with borders and annotations
#Calculate bin edges and frequencies
plt.figure(figsize=(12, 6))
hist, bin_edges = np.histogram(death_df['Age'], bins=bins)


plt.hist(death_df['Age'], bins=bins, edgecolor='black')
plt.xlabel("Age", labelpad=30)
plt.ylabel("Frequency")
plt.title("Age Histogram")
plt.xticks([])

# Annotate bin ranges and Age_Bin values on x-axis (horizontal)
bin_ranges = [(int(bin_edges[i]), int(bin_edges[i + 1])) for i in range(len(bin_edges) - 1)]
x_positions = [(bin_edges[i] + bin_edges[i + 1]) / 2 for i in range(len(bin_edges) - 1)]

for i, (bin_range, x_pos) in enumerate(zip(bin_ranges, x_positions)):
    plt.text(x_pos, -20, f"{bin_range[0]}-{bin_range[1]}", ha='center', rotation=0)
    plt.text(x_pos, -40, f"Age_Bin: {i}", ha='center', rotation=0)

# Annotate frequencies on top of bars
for i, freq in enumerate(hist):
    plt.text(x_positions[i], freq + 5, str(freq), ha='center', va='bottom')

plt.show()

# Drop the redundant column
death_df = death_df.drop(columns=['Age'])
death_df = death_df.drop(columns=['Date Type'])


# 3.4 Merging Similar Categories in 'Race' Column and Encoding

In [ ]:
# Simplify race
# Convert the 'Race' column back to its original string representation before applying the lambda function
death_df['Race'] = death_df['Race'].astype(str)
death_df['Race'] = death_df['Race'].apply(lambda race: 'White' if 'White' in race else ('Black' if 'Black' in race or 'Black or African American' in race else race))

# CREATing MAPPINGS BEFORE ENCODING!
# Race
race_mapping = dict(enumerate(death_df['Race'].astype('category').cat.categories))
# Residence City
residence_city_mapping = dict(enumerate(death_df['Residence City'].astype('category').cat.categories))
# Death City
death_city_mapping = dict(enumerate(death_df['Death City'].astype('category').cat.categories))

# 3.5 Encoding 'Residence City' column and Dropping 'Death City' column

Due to high correlation skewing results

In [ ]:
# Simplify and Encode Residence City and Death City
death_df['Residence City'] = death_df['Residence City'].astype(str)
death_df['Residence City'] = death_df['Residence City'].apply(lambda city: city if city in allowed_residence_cities else 'Other')
death_df['Death City'] = death_df['Death City'].astype(str)
death_df['Death City'] = death_df['Death City'].apply(lambda city: city if city in allowed_death_cities else 'Other')

# Encode Residence City
death_df['Residence City'] = death_df['Residence City'].astype('category').cat.codes

# Encode Death City
death_df['Death City'] = death_df['Death City'].astype('category').cat.codes

# Encode Race
death_df['Race'] = death_df['Race'].astype('category').cat.codes

# Race
print("Race:", race_mapping)

# Residence City
print("Residence City:", residence_city_mapping)

# Death City
print("Death City:", death_city_mapping)

# Drop the 'Death City' column due to too much correlation with residence city
death_df = death_df.drop(columns=['Residence City'])



# 3.6 Encoding Relevant Drug Indicator columns

In [ ]:
# Handles encoding drug columns
for col in ['Heroin', 'Cocaine', 'Fentanyl', 'Other Opioid']:
    if col in death_df.columns:
        # Replace 'Y' with 1 and empty strings with 0 before converting to int
        death_df[col] = death_df[col].replace({'Y': 1, '': 0}).fillna(0).astype(int)

# 3.7 Filling in any missing values and one-hot encoding multiclass columns

In [ ]:
# Before applying Apriori, we must any remaining fill NaN values with 0
death_df = death_df.fillna(0)

# Select columns for one-hot encoding
multiclass_cols = ['Age_Bin', 'Race', 'Death City']


# Perform one-hot encoding
encoded_df = pd.get_dummies(death_df, columns=multiclass_cols, prefix=multiclass_cols, prefix_sep='_', dummy_na=False, drop_first=False) #drop_first = False to keep all classes

# Print all features (column names) and the number of 1s
print("Features after one-hot encoding and number of values with 1:")
for column in encoded_df.columns:
    num_ones = encoded_df[column].sum()  # Calculate the sum of 1s in the column
    print(f"{column}: {num_ones}")


# Before applying Apriori, fill any potential NaN values (after encoding} with 0
encoded_df = encoded_df.fillna(0)

# Select columns with binary values (0 or 1) for apriori/fpgrowth (columns that were converted + existing binary columns)
apriori_cols = ['Sex', 'Heroin', 'Cocaine', 'Fentanyl', 'Other Opioid'] + [col for col in encoded_df.columns if col not in death_df.columns]
apriori_df = encoded_df[apriori_cols]


# 3.8 Preprocessing Complete - Understanding the Final Dataframe

In [ ]:
#Gettting an idea for the new unique values in each feature
for column in death_df.columns:
    print(f"**Column:** `{column}`")
    value_counts = death_df[column].value_counts()
    formatted_values = [f"{value} ({count})" for value, count in value_counts.items()]
    print("  Unique Values:", ", ".join(formatted_values))

    print("-" * 40)
print("Shape of final DataFrame is: ", death_df.shape)

# 4. Implementing Apriori

# 4.1 Checking for Strongly Associated Features or Featuresets

In [ ]:
from mlxtend.frequent_patterns import apriori as ap
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder


# Define minimum support and confidence thresholds
min_support = 0.03  # Meaning thhis Itemset must appear in at least 3% of cases
min_confidence = 0.70  # So Rule must be correct at minimum 70% of the time

# Convert data for Apriori using TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(apriori_df.astype(bool).values).transform(apriori_df.astype(bool).values)
apriori_df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# Record start time
start_time_apriori = time.time()

# Apply Apriori algorithm to find frequent itemsets
frequent_itemsets_apriori = ap(apriori_df, min_support=min_support, use_colnames=True)
rules_apriori = association_rules(frequent_itemsets_apriori, metric="confidence", min_threshold=min_confidence)

# Checking frequent random itemsets
frequent_itemsets_apriori['proportion'] = np.random.rand(len(frequent_itemsets_apriori))
frequent_itemsets_apriori = frequent_itemsets_apriori.sort_values(by='proportion')
print(frequent_itemsets_apriori.head(5), "\n")

rules_apriori = association_rules(frequent_itemsets_apriori, metric="confidence", min_threshold=min_confidence)

# Filter rules to get meaningful associations
filtered_rules = rules_apriori[
    (rules_apriori['antecedents'].apply(lambda x: not (len(x) == 1 and list(x)[0] in ['Sex', 'Race_1', 'Death City_0']))) &
    (rules_apriori['consequents'].apply(lambda x: not (len(x) == 1 and list(x)[0] in ['Sex', 'Race_1', 'Death City_0']))) &
    (rules_apriori['lift'] > 1)
]

# Introduce metric that NCLF to see how rules compare based on all three
filtered_rules['normalized_confidence_lift_ratio'] = filtered_rules['confidence'] * filtered_rules['lift'] / filtered_rules['support']
print(filtered_rules.head(5))

# Record end time and Calculate execution time
end_time_apriori = time.time()
execution_time_apriori = end_time_apriori - start_time_apriori
print(f"Apriori Execution Time: {execution_time_apriori:.4f} seconds")

# Display filtered association rules
print("\nAssociation Rules (Apriori - Filtered):")
print(filtered_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

# Sort rules by highest lift value (strongest associations)
sorted_rules_apriori = filtered_rules.sort_values(by="lift", ascending=False)

# Display top 5 strongest rules
print("\nTop 5 Strongest Association Rules (Apriori - Filtered):")
print(sorted_rules_apriori.head(5))


**Run 5-6 times until execution time stabilizes**

**Encoded Value Key:**

**Race**:
 * Race_0: Black
 * Race_1: White

**Sex**:
 * 0: Female
 * 1: Male

**Age_Bin**:
*  Age_Bin_0.0: 17-27
*  Age_Bin_1.0: 27-37
*  Age_Bin_2.0: 37-47
*  Age_Bin_3.0: 47-57
*  Age_Bin_4.0: 57-67
*  Age_Bin_5.0: 67-77
*  Age_Bin_6.0: 77+



**Death City**:
*  Death City_0: BRIDGEPORT
*  Death City_1: HARTFORD
*  Death City_2: NEW BRITAIN
*  Death City_3: NEW HAVEN
*  Death City_4: WATERBURY


# 4.2 Checking for Negatively Associated Features or Featuresets

Lift below 1 (with good support and confidence) means the presence of the antecedent reduces the likehood (in this case by 11-14%) of the consequent being present

In [ ]:
# Apply Apriori algorithm to find frequent itemsets (without minimum support)
frequent_itemsets_apriori_all = ap(apriori_df, min_support=0.12, use_colnames=True)  # Occurs atleast 12% of the time

# Generate association rules (without minimum confidence)
rules_apriori_all = association_rules(frequent_itemsets_apriori_all, metric="lift", min_threshold=0)  # Lift must be valid

# Filter rules to exclude those where 'Sex' or 'Race_1' are the only antecedent or consequent
filtered_rules_all = rules_apriori_all[
    rules_apriori_all['antecedents'].apply(lambda x: len(x) > 1 or (len(x) == 1 and list(x)[0] not in ['Sex', 'Race_1'])) &
    rules_apriori_all['consequents'].apply(lambda x: len(x) > 1 or (len(x) == 1 and list(x)[0] not in ['Sex', 'Race_1']))
]

# Filter for lift <= 1
filtered_rules_all = filtered_rules_all[filtered_rules_all['lift'] <= 0.9] #Filtering for negative association


# Sort rules by confidence in descending order
sorted_rules_apriori_lowest = filtered_rules_all.sort_values(by="confidence", ascending=False)  #With best confidence

# Display top 5 weakest rules
print("\nTop 5 Weakest (Negative) Association Rules (Apriori - Filtered):")
print(sorted_rules_apriori_lowest.head(5))

# 5. Implementing FPgrowth

In [ ]:
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules
import time

min_confidence = 0.60  # So Rule must be correct at minimum 60% of the time
min_support=0.15 # So feature sets appear at least 15% of the time

# Prune rare columns and rows for improved performance
apriori_df_dropped = apriori_df.drop(columns=['Other Opioid'], errors='ignore')

# Record start time
start_time_fpgrowth = time.time()

# Apply FP-Growth algorithm to find frequent itemsets (pruning low freq items) #low_memory parameter was attempted but is not accesible in this version of the library
frequent_itemsets_fpgrowth = fpgrowth(apriori_df_dropped, min_support=0.15, use_colnames=True, max_len=None)
rules_fpgrowth = association_rules(frequent_itemsets_fpgrowth, metric="confidence", min_threshold=min_confidence)

# Record end time and Calculate execution time
end_time_fpgrowth = time.time()
execution_time_fpgrowth = end_time_fpgrowth - start_time_fpgrowth

print(f"FP-Growth Execution Time: {execution_time_fpgrowth:.4f} seconds")

# Filter rules to exclude those where 'Sex' or 'Race_1' are the only antecedent or consequent
filtered_rules_fpgrowth = rules_fpgrowth[
    rules_fpgrowth['antecedents'].apply(lambda x: len(x) > 1 or (len(x) == 1 and list(x)[0] not in ['Sex', 'Race_1'])) &
    rules_fpgrowth['consequents'].apply(lambda x: len(x) > 1 or (len(x) == 1 and list(x)[0] not in ['Sex', 'Race_1']))
]

# Display filtered association rules
print("\nAssociation Rules (FP-Growth - Filtered):")
print(filtered_rules_fpgrowth[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

# Then, Sort rules by highest lift value (strongest associations)
sorted_rules_fpgrowth = filtered_rules_fpgrowth.sort_values(by="lift", ascending=False)

# Finally, Display top 3 strongest rules
print("\nTop 3 Strongest Association Rules (FP-Growth - Filtered):")
print(sorted_rules_fpgrowth.head(3))

**Run 5-6 times until execution time stabilizes**

# 6. Visualizing Common associations and Featuresets

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Creating my directed graph (DiGraph)
G = nx.DiGraph()

# 2. Adding nodes
for itemset in frequent_itemsets_fpgrowth:
    for item in itemset:
        G.add_node(item)

# 3. Adding the edges (rules) with lift as weight
for index, row in filtered_rules_fpgrowth.iterrows():
    antecedents = list(row['antecedents'])
    consequents = list(row['consequents'])
    support = row['support']
    lift = row['lift']
    confidence = row['confidence']

    for antecedent in antecedents:
        for consequent in consequents:
            G.add_edge(antecedent, consequent, weight=7**(lift))

# 4. Finding the largest weakly connected component in my directed graphs
largest_cc = max(nx.weakly_connected_components(G), key=len)

# 5. Writing a subgraph with only the largest connected component
subgraph = G.subgraph(largest_cc)

# 6. Draw the subgraph with edge directions
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(subgraph)
nx.draw(subgraph, pos, with_labels=True, node_size=2000, node_color="skyblue", font_size=10,
        width=[subgraph[u][v]['weight'] * 0.6 for u, v in subgraph.edges],
        arrowsize=20)  # arrowsize for better visibility
plt.title("FP Growth Association Rule Network Graph - Weighted by Lift (Directed)")
plt.show()

Just for Checking

In [ ]:
#Gettting an idea for the new unique values in each feature data mining adjusted Data Structure
for column in apriori_df.columns:
    print(f"**Column:** `{column}`")
    value_counts = apriori_df[column].value_counts()
    formatted_values = [f"{value} ({count})" for value, count in value_counts.items()]
    print("  Unique Values:", ", ".join(formatted_values))

    print("-" * 40)